# Notebook 04 — Grad-CAM Visualisation

**Experiment 4** from the proposal.

Grad-CAM (Selvaraju et al., 2017 — Lecture 11) lets us see *where* in the chart
the CNN looks when making its prediction.  We should expect the model to focus on:

- The peaks / troughs that define the pattern (e.g., the head in Head & Shoulders)
- The breakout region at the right side of the chart

If Grad-CAM highlights random noise instead, it signals the model learned spurious
correlations rather than true pattern features.


In [ ]:
import sys, os
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms

from config import *
from src.models.cnn                 import PatternCNN
from src.data_generation.patterns   import generate_pattern_sample
from src.data_generation.chart_renderer import render_ohlc_to_pil
from src.visualization.gradcam      import GradCAM, visualize_gradcam, gradcam_grid
from src.training.dataset            import get_transforms

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 1. Load trained model

In [ ]:
CKPT = os.path.join(ROOT, 'checkpoints', 'cnn_best.pt')

model = PatternCNN(num_classes=7)
ckpt  = torch.load(CKPT, map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
model.to(DEVICE).eval()
print(f'Model loaded — val_acc = {ckpt["val_acc"]:.4f}')

## 2. Single-image Grad-CAM — one per class

In [ ]:
transform = get_transforms(train=False, img_size=IMG_SIZE)

# Hook into the last convolutional block of PatternCNN
# block3 is a _ConvBlock; its first child is the Conv2d layer
target_layer = list(model.block3.block.children())[0]   # Conv2d in block3

for class_id in range(len(CLASS_NAMES)):
    ohlc     = generate_pattern_sample(class_id, n_points=N_CANDLES)
    pil_img  = render_ohlc_to_pil(ohlc, img_size=IMG_SIZE)
    tensor   = transform(pil_img).unsqueeze(0).to(DEVICE)

    with GradCAM(model, target_layer) as gcam:
        heatmap, pred, conf = gcam.compute(tensor, class_idx=class_id)

    fig = visualize_gradcam(
        pil_img, heatmap, pred, conf,
        class_names=CLASS_NAMES,
        save_path=os.path.join(ROOT, 'results', 'figures',
                               f'gradcam_{CLASS_NAMES[class_id]}.png'),
    )
    plt.show()
    print()

## 3. Multi-sample Grad-CAM grid — Head & Shoulders vs No Pattern

In [ ]:
for class_id in [0, 6]:   # Head&Shoulders vs No Pattern
    images = [render_ohlc_to_pil(generate_pattern_sample(class_id, n_points=N_CANDLES),
                                  img_size=IMG_SIZE)
              for _ in range(5)]

    fig = gradcam_grid(
        model, target_layer, images, transform, DEVICE,
        class_names=CLASS_NAMES,
        save_path=os.path.join(ROOT, 'results', 'figures',
                               f'gradcam_grid_{CLASS_NAMES[class_id]}.png'),
    )
    plt.suptitle(f'Grad-CAM grid — {CLASS_NAMES[class_id]}', fontsize=12, y=1.02)
    plt.show()

## 4. Grad-CAM on real stock windows

Load a real window the model detected with high confidence and visualise what it focused on.

In [ ]:
import pandas as pd
from src.verification.stock_data import _download, _ohlc_window_to_image

# Load saved detections from notebook 03
det_path = os.path.join(ROOT, 'results', 'logs', 'detections.csv')
if not os.path.exists(det_path):
    print('Run notebook 03 first to generate detections.csv')
else:
    det_df = pd.read_csv(det_path)

    # Pick the 4 most confident non-trivial detections
    top = (det_df[det_df['class_name'] != 'no_pattern']
           .sort_values('confidence', ascending=False)
           .head(4)
           .reset_index(drop=True))

    for _, row in top.iterrows():
        ticker = row['ticker']
        date   = pd.Timestamp(row['date'])

        ohlc = _download(ticker, start='2015-01-01', end='2024-12-31')
        if ohlc is None:
            continue

        # Find the window ending on that date
        try:
            end_idx   = ohlc.index.get_loc(date, method='nearest')
        except Exception:
            continue
        start_idx = max(0, end_idx - WINDOW_SIZE + 1)
        window    = ohlc.iloc[start_idx : end_idx + 1]

        if len(window) < 30:
            continue

        pil_img = _ohlc_window_to_image(window, img_size=IMG_SIZE)
        tensor  = transform(pil_img).unsqueeze(0).to(DEVICE)

        with GradCAM(model, target_layer) as gcam:
            heatmap, pred, conf = gcam.compute(tensor)

        fig = visualize_gradcam(
            pil_img, heatmap, pred, conf, class_names=CLASS_NAMES,
            save_path=os.path.join(ROOT, 'results', 'figures',
                                   f'gradcam_real_{ticker}_{date.date()}.png'),
        )
        plt.suptitle(f'{ticker}  {date.date()}  |  true label: {row["class_name"]}',
                     fontsize=10)
        plt.show()